In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

from sklearn.linear_model import Ridge
from sklearn.ensemble import AdaBoostRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv(
    r"D:\Vehicle_Mileage_Project\data\processed\cleaned_dataset.csv"
)

df.head()

,Vehicle_ID,Manufacturer,Model,Vehicle_Age,Engine_CC,Horsepower,Vehicle_Weight,Transmission,Fuel_Type,Ethanol_Blend,...,Traffic_Level,Temperature,Humidity,Weather,AC_Usage,Driving_Style,Passengers,CO2_Emission_g_km,Fuel_Consumption_L_100km,Mileage_km_per_L
0,V014767,Tata,Nexon,13.0,1000.0,82.0,1307.0,Automatic,Petrol,20.0,...,High,27.0,61.0,Fog,No,Normal,1.0,192.1,5.00,20.02
1,V020736,Toyota,Glanza,8.0,1200.0,98.0,1493.0,Automatic,Petrol,20.0,...,High,30.0,48.0,Fog,Yes,Smooth,3.0,189.2,5.12,19.52
2,V041583,Maruti,Baleno,9.0,1200.0,103.0,1262.0,Automatic,Petrol,20.0,...,Medium,34.0,44.0,Fog,Yes,Aggressive,1.0,197.3,5.55,18.01
3,V042258,Honda,City,14.0,1200.0,95.0,1001.0,Manual,Petrol,10.0,...,High,15.0,53.0,Rain,Yes,Normal,1.0,185.8,5.23,19.12
4,V029600,Tata,Altroz,15.0,1800.0,148.0,1604.0,Manual,Petrol,10.0,...,Medium,24.0,65.0,Fog,Yes,Smooth,1.0,197.7,5.90,16.96


In [3]:
df.drop(
    columns=[
        "Vehicle_ID",
        "Manufacturer",
        "Model",
        "Fuel_Type",
        "CO2_Emission_g_km",
        "Fuel_Consumption_L_100km"
    ],
    inplace=True
)

In [4]:
X = df.drop("Mileage_km_per_L", axis=1)

y = df["Mileage_km_per_L"]

In [5]:
categorical_columns = X.select_dtypes(include=["object", "string"]).columns

categorical_columns

Index(['Transmission', 'Maintenance_Status', 'Road_Type', 'Traffic_Level',
       'Weather', 'AC_Usage', 'Driving_Style'],
      dtype='str')

In [6]:
encoders = {}

for col in categorical_columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(X[col])

    encoders[col] = le

In [7]:
print(encoders["Transmission"].classes_)

['Automatic' 'Manual']


In [8]:
joblib.dump(
    encoders,
    r"D:\Vehicle_Mileage_Project\models\label_encoders.pkl"
)

['D:\\Vehicle_Mileage_Project\\models\\label_encoders.pkl']

In [9]:
joblib.dump(
    X.columns.tolist(),
    r"D:\Vehicle_Mileage_Project\models\feature_names.pkl"
)

['D:\\Vehicle_Mileage_Project\\models\\feature_names.pkl']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
ada = AdaBoostRegressor(random_state=42)

ada.fit(X_train, y_train)

ada_pred = ada.predict(X_test)

In [12]:
print("AdaBoost")

print("R2 :", r2_score(y_test, ada_pred))
print("MAE:", mean_absolute_error(y_test, ada_pred))
print("RMSE:", root_mean_squared_error(y_test, ada_pred))

AdaBoost
R2 : 0.8377113620500739
MAE: 1.127386088983403
RMSE: 1.4066962687435915


In [13]:
joblib.dump(
    ada,
    r"D:\Vehicle_Mileage_Project\models\adaboost.pkl"
)

['D:\\Vehicle_Mileage_Project\\models\\adaboost.pkl']

In [14]:
ridge = Ridge()

ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)

In [15]:
print("Ridge")

print("R2 :", r2_score(y_test, ridge_pred))
print("MAE:", mean_absolute_error(y_test, ridge_pred))
print("RMSE:", root_mean_squared_error(y_test, ridge_pred))

Ridge
R2 : 0.7619801661629984
MAE: 1.3930304753465763
RMSE: 1.703581250701337


In [16]:
joblib.dump(
    ridge,
    r"D:\Vehicle_Mileage_Project\models\ridge_regression.pkl"
)

['D:\\Vehicle_Mileage_Project\\models\\ridge_regression.pkl']

In [17]:
cat = CatBoostRegressor(
    verbose=0,
    random_state=42
)

cat.fit(X_train, y_train)

cat_pred = cat.predict(X_test)

In [18]:
print("CatBoost")

print("R2 :", r2_score(y_test, cat_pred))
print("MAE:", mean_absolute_error(y_test, cat_pred))
print("RMSE:", root_mean_squared_error(y_test, cat_pred))

CatBoost
R2 : 0.971477674383208
MAE: 0.5085074932043091
RMSE: 0.5897239212282697


In [19]:
joblib.dump(
    cat,
    r"D:\Vehicle_Mileage_Project\models\catboost.pkl"
)

['D:\\Vehicle_Mileage_Project\\models\\catboost.pkl']

In [20]:
param_grid = {
    "iterations": [300, 500, 700],
    "learning_rate": [0.03, 0.05],
    "depth": [4, 6, 8]
}

grid = GridSearchCV(
    estimator=CatBoostRegressor(
        verbose=0,
        random_state=42
    ),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","CatBoostRegre...42, verbose=0)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'depth': [4, 6, ...], 'iterations': [300, 500, ...], 'learning_rate': [0.03, 0.05]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_d

In [27]:
grid_best = grid.best_estimator_

grid_pred = grid_best.predict(X_test)

print("GridSearchCV")
print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

print("\nTest Performance")
print("R2 :", r2_score(y_test, grid_pred))
print("MAE:", mean_absolute_error(y_test, grid_pred))
print("RMSE:", root_mean_squared_error(y_test, grid_pred))

GridSearchCV
Best Parameters: {'depth': 4, 'iterations': 700, 'learning_rate': 0.05}
Best CV Score: 0.9721083269337181

Test Performance
R2 : 0.9720696054427533
MAE: 0.5043934780460048
RMSE: 0.5835724932893193


In [28]:
import joblib

joblib.dump(
    best_model,
    r"D:\Vehicle_Mileage_Project\models\catboost_gridsearch.pkl"
)

['D:\\Vehicle_Mileage_Project\\models\\catboost_gridsearch.pkl']

In [29]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "iterations": [300,500,700,900,1100],
    "learning_rate": [0.01,0.03,0.05,0.1],
    "depth": [4,5,6,7,8],
    "l2_leaf_reg": [3,5,7,9]
}

random = RandomizedSearchCV(
    estimator=CatBoostRegressor(
        verbose=0,
        random_state=42
    ),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random.fit(X_train, y_train)

random_best = random.best_estimator_

random_pred = random_best.predict(X_test)

print("RandomizedSearchCV")
print("Best Parameters :", random.best_params_)
print("Best CV Score :", random.best_score_)
print("R2 :", r2_score(y_test, random_pred))
print("MAE :", mean_absolute_error(y_test, random_pred))
print("RMSE :", root_mean_squared_error(y_test, random_pred))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
RandomizedSearchCV
Best Parameters : {'learning_rate': 0.03, 'l2_leaf_reg': 3, 'iterations': 700, 'depth': 4}
Best CV Score : 0.9720879016400016
R2 : 0.9721317266405283
MAE : 0.5038343036064121
RMSE : 0.5829231576313731


In [30]:
joblib.dump(
    random_best,
    r"D:\Vehicle_Mileage_Project\models\catboost_randomsearch.pkl"
)

['D:\\Vehicle_Mileage_Project\\models\\catboost_randomsearch.pkl']

In [31]:
results = pd.DataFrame({
    "Method": ["GridSearchCV", "RandomizedSearchCV"],
    "CV Score": [
        grid.best_score_,
        random.best_score_
    ],
    "Test R2": [
        r2_score(y_test, grid_pred),
        r2_score(y_test, random_pred)
    ],
    "MAE": [
        mean_absolute_error(y_test, grid_pred),
        mean_absolute_error(y_test, random_pred)
    ],
    "RMSE": [
        root_mean_squared_error(y_test, grid_pred),
        root_mean_squared_error(y_test, random_pred)
    ]
})

results

,Method,CV Score,Test R2,MAE,RMSE
0,GridSearchCV,0.972108,0.972070,0.504393,0.583572
1,RandomizedSearchCV,0.972088,0.972132,0.503834,0.582923


In [34]:
if r2_score(y_test, grid_pred) >= r2_score(y_test, random_pred):
    best_model = grid_best
    print("GridSearchCV model selected as Best Model")
else:
    best_model = random_best
    print("RandomizedSearchCV model selected as Best Model")

RandomizedSearchCV model selected as Best Model


In [35]:
import joblib

joblib.dump(
    best_model,
    r"D:\Vehicle_Mileage_Project\models\best_model.pkl"
)

print("best_model.pkl saved successfully.")

best_model.pkl saved successfully.
